In [ ]:
"""
07c - Upload the result rasters to Earth Engine for visualisation

Uploads the five rasters written by notebook 07b as Earth Engine Image assets, so they
can be inspected against a Sentinel-2 base map in the Code Editor.

Prerequisite: notebook 07b has completed and these files exist under
outputs/inference_full_roi/:
    severity_30m.tif  persistence_30m.tif  recovery_gap_30m.tif
    recurrence_30m.tif  burned_prob_30m.tif

This step is for visual inspection only. It produces no thesis result.
Runtime: 10 to 25 minutes, depending on file size and connection.
"""

'\nNotebook 07c v4 — Upload result rasters to GEE for visualisation\n\nPurpose:\n    Guide the upload of the 5 result rasters from Notebook 07b to GEE\n    as Image assets so they can be visualised together with a Sentinel-2\n    base map in the GEE Code Editor.\n\nPrerequisites:\n    Notebook 07b v4 has completed. The following files exist in\n    MyDrive/thesisv4/outputs/inference_full_roi/:\n        - severity_30m.tif\n        - persistence_30m.tif\n        - recovery_gap_30m.tif\n        - recurrence_30m.tif\n        - burned_prob_30m.tif       (new in v3)\n\nRuntime: 10-25 minutes (upload depends on file size and connection).\n'

# 07c - Upload results to Earth Engine

**Purpose** - publish the 30 m result rasters as Earth Engine assets for visual
inspection against a Sentinel-2 base map.

**Main inputs** - the rasters written by notebook 07b.

**Main outputs** - Earth Engine Image assets and `gee_asset_info.json`.

**Role in the thesis workflow** - visualisation only. No thesis number, table or
figure depends on this notebook; it is included so the 30 m chain is complete.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import sys
import ee
import json
import time
from pathlib import Path
from datetime import datetime

# The Earth Engine project id is not committed. gee_project reads it from
# the GEE_PROJECT environment variable or from configs/gee_config.json.
THESIS_ROOT = os.environ.get("THESIS_ROOT", "/content/drive/MyDrive/thesisv4")
sys.path.insert(0, THESIS_ROOT)
import gee_project

PROJECT = gee_project.require()

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("GEE initialised.")

GEE initialised.


## 2. Locate the result rasters

In [ ]:
RESULTS_DIR = Path("/content/drive/MyDrive/thesisv4/outputs/inference_full_roi")

result_files = {
    "severity":     RESULTS_DIR / "severity_30m.tif",
    "persistence":  RESULTS_DIR / "persistence_30m.tif",
    "recovery_gap": RESULTS_DIR / "recovery_gap_30m.tif",
    "recurrence":   RESULTS_DIR / "recurrence_30m.tif",
    "burned_prob":  RESULTS_DIR / "burned_prob_30m.tif",
}

for name, path in result_files.items():
    assert path.exists(), f"Missing: {path}"
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"  {name:15s} {path.name} ({size_mb:.1f} MB)")

  severity        severity_30m.tif (16.1 MB)
  persistence     persistence_30m.tif (16.1 MB)
  recovery_gap    recovery_gap_30m.tif (16.1 MB)
  recurrence      recurrence_30m.tif (16.1 MB)
  burned_prob     burned_prob_30m.tif (16.1 MB)


## 3. Prepare asset paths

In [ ]:
TIMESTAMP    = datetime.now().strftime("%Y%m%d_%H%M")
ASSET_PREFIX = f"projects/{PROJECT}/assets/ris_inference_v4_{TIMESTAMP}"

print(f"Assets will be named: {ASSET_PREFIX}_<dimension>")

Assets will be named: projects/YOUR-GEE-PROJECT/assets/ris_inference_v4_20260813_0934_<dimension>


## 4. Manual upload - the recommended route

The Python API has no direct upload-from-Drive function, so the practical route is
manual, through the Earth Engine Code Editor:

1. Open the Code Editor at https://code.earthengine.google.com/
2. Open the **Assets** tab on the left
3. Choose **NEW - Image upload (GeoTIFF)**
4. Select the file from Drive (download it first if the browser upload dialog does
   not offer a "from Drive" option)
5. Set the asset name to the value listed below

In [ ]:
print("Manual upload paths (files are on Drive):")
for name, path in result_files.items():
    print(f"  {name:15s}: {path}")

print("\nAsset names to use in the GEE upload dialog:")
for name in result_files:
    print(f"  {ASSET_PREFIX}_{name}")

## 5. Alternative - automated upload through Cloud Storage

If a Cloud Storage bucket is available to the project, this cell uploads each raster
to the bucket and ingests it as an Earth Engine asset. Skip it if you are using the
manual route above.

In [ ]:
# Uncomment and configure if using GCS:
#
# GCS_BUCKET = "your-bucket-name"
#
# from google.cloud import storage
# client = storage.Client(project=PROJECT)
# bucket = client.bucket(GCS_BUCKET)
#
# for name, local_path in result_files.items():
#     gcs_blob = f"thesis_uploads/{name}_30m_{TIMESTAMP}.tif"
#     blob = bucket.blob(gcs_blob)
#     print(f"Uploading {name} to gs://{GCS_BUCKET}/{gcs_blob}...")
#     blob.upload_from_filename(local_path)
#
#     asset_id   = f"{ASSET_PREFIX}_{name}"
#     request_id = ee.data.newTaskId()[0]
#     params = {
#         "id":      asset_id,
#         "sources": [{"uris": [f"gs://{GCS_BUCKET}/{gcs_blob}"]}],
#     }
#     ee.data.startIngestion(request_id, params)
#     print(f"  Ingestion started for {asset_id}")

## 6. Save the asset metadata for the visualisation script

In [ ]:
asset_info = {
    "timestamp":    TIMESTAMP,
    "asset_prefix": ASSET_PREFIX,
    "expected_assets": [f"{ASSET_PREFIX}_{name}"
                        for name in result_files.keys()],
}

with open(RESULTS_DIR / "gee_asset_info.json", "w") as f:
    json.dump(asset_info, f, indent=2)

print(json.dumps(asset_info, indent=2))
print(f"\nSaved to: {RESULTS_DIR / 'gee_asset_info.json'}")
print("\nAfter uploading, paste the JS visualisation script into the GEE")
print("Code Editor and update TIMESTAMP to match the value above.")

{
  "timestamp": "20260813_0934",
  "asset_prefix": "projects/YOUR-GEE-PROJECT/assets/ris_inference_v4_20260813_0934",
  "expected_assets": [
    "projects/YOUR-GEE-PROJECT/assets/ris_inference_v4_20260813_0934_severity",
    "projects/YOUR-GEE-PROJECT/assets/ris_inference_v4_20260813_0934_persistence",
    "projects/YOUR-GEE-PROJECT/assets/ris_inference_v4_20260813_0934_recovery_gap",
    "projects/YOUR-GEE-PROJECT/assets/ris_inference_v4_20260813_0934_recurrence",
    "projects/YOUR-GEE-PROJECT/assets/ris_inference_v4_20260813_0934_burned_prob"
  ]
}

Saved to: /content/drive/MyDrive/thesisv4/outputs/inference_full_roi/gee_asset_info.json

After uploading, paste the JS visualisation script into the GEE
Code Editor and update TIMESTAMP to match the value above.
